# TPU lane: is it reachable, and what is on it

Reachability and device facts only. The TPU comparison design is separate; this
kernel exists so the lane is known to work before anything is built on it, and
it costs seconds of the ~20-30 TPU-hours a week.

No payload dataset is attached and nothing is installed: the answers come from
`jax`, `libtpu`, and the accelerator host devices that Kaggle exposes.

In [ ]:
import os, platform, subprocess, sys

OUT = "/kaggle/working"
INPUT = "/kaggle/input"
LOGD = os.path.join(OUT, "logs")
os.makedirs(LOGD, exist_ok=True)
FACTS = {}          # filled in below; later cells read it


def sh(cmd, timeout=120):
    """Run a shell command, return its combined output, never raise."""
    try:
        p = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)
        return (p.stdout + p.stderr).strip()
    except subprocess.TimeoutExpired:
        return f"<timeout after {timeout}s>"
    except Exception as e:
        return f"<{type(e).__name__}: {e}>"


def sh_ok(cmd, timeout=120):
    """Like sh(), but empty unless the command SUCCEEDED.

    Needed because "nvidia-smi: command not found" on stderr is non-empty
    output from a command that failed, and counting it as a GPU is exactly the
    kind of lie this cell exists to prevent.
    """
    try:
        p = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)
        return (p.stdout + p.stderr).strip() if p.returncode == 0 else ""
    except Exception:
        return ""


def say(*a):
    print(*a, flush=True)


def fact(k, v):
    FACTS[k] = v
    return v


# ---- what kind of machine is this? decided by evidence, not by the metadata
gpu_names = sh_ok("nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv,noheader")
n_gpu = len(gpu_names.splitlines())
tpu_lib = sh_ok("ls -1 /dev/accel* 2>/dev/null; ls -1 /usr/lib/libtpu* /usr/lib/x86_64-linux-gnu/libtpu* 2>/dev/null")

kind = "CPU"
if n_gpu > 0:
    kind = "GPU"
elif tpu_lib:
    # a TPU runner has no working nvidia-smi, but the accelerator host devices
    # and libtpu are both there. jax is asked next; this is the cheap check that
    # runs before paying for a jax import.
    kind = "TPU"
fact("kind", kind)

say("=" * 78)
say("device")
say("=" * 78)
say(f"  python      {sys.version.split()[0]}  {platform.platform()}")
say(f"  cwd         {os.getcwd()}")
say(f"  kaggle input{os.path.isdir(INPUT)}  "
    f"{sorted(os.listdir(INPUT)) if os.path.isdir(INPUT) else '(nothing attached)'}")
say(f"  nvidia-smi  {gpu_names or 'unavailable'}")
say(f"  tpu nodes   {tpu_lib or 'unavailable'}")
say(f"  ==> this is a {kind} runner")

env_keys = ("KAGGLE_", "CUDA_", "NVIDIA_", "TPU_", "PJRT_", "XLA_", "LD_LIBRARY_PATH")
with open(os.path.join(LOGD, "00-environment.txt"), "w") as f:
    f.write(f"# {platform.platform()} python {sys.version.split()[0]}\n")
    f.write(f"# nvidia-smi: {gpu_names or 'unavailable'}\n")
    f.write(f"# tpu: {tpu_lib or 'unavailable'}\n")
    for k in sorted(os.environ):
        if k.startswith(env_keys):
            line = f"{k}={os.environ[k]}"
            say("  env " + line[:150])
            f.write(line + "\n")
say(f"  disk        {sh('df -h ' + OUT + ' | tail -1')}")
say(f"  cpus        {os.cpu_count()}")

In [ ]:
import os, shutil, subprocess, sys

OUT = "/kaggle/working"
LOGD = os.path.join(OUT, "logs")
FACTS = {}
FACTS["kind"] = "TPU"
FACTS["gpu"] = ""
FACTS["payload_sha"] = "n/a (none attached)"
FACTS["versions"] = ""

say("")
say("=" * 78)
say("TPU device facts")
say("=" * 78)

# ---- the host side
for label, cmd in (
    ("accel devices", "ls -l /dev/accel* 2>&1 | head"),
    ("libtpu", "ls -1 /usr/lib/libtpu* /usr/lib/x86_64-linux-gnu/libtpu* 2>&1 | head"),
    ("TPU env", "env | grep -iE '^(TPU|PJRT|XLA)' | sort"),
    ("nvidia (should be empty)", "nvidia-smi -L 2>&1 | head -2"),
):
    say(f"  {label:<26} " + sh(cmd, timeout=120).replace("\n", "\n  " + " " * 28))

# ---- jax: the question is not "is there a TPU" but "does jax see it"
try:
    import jax
    devs = jax.devices()
    FACTS["versions"] = f"jax {jax.__version__}"
    say(f"  {'jax':<26} {jax.__version__}")
    say(f"  {'jax.default_backend':<26} {jax.default_backend()}")
    say(f"  {'device count':<26} {jax.device_count()}")
    for d in devs:
        say(f"  {'device':<26} kind={d.device_kind} platform={d.platform} "
            f"id={getattr(d, 'id', '?')} process={getattr(d, 'process_index', '?')}")
        try:
            ms = d.memory_stats() or {}
            hbm = ms.get("bytes_limit") or ms.get("bytes_in_use")
            if hbm:
                say(f"  {'  memory_stats':<26} {hbm/1e9:.1f} GB  {ms}")
            else:
                say(f"  {'  memory_stats':<26} {ms or '(empty)'}")
        except Exception as e:
            say(f"  {'  memory_stats':<26} unsupported ({type(e).__name__}: {e})")
    FACTS["jax_devices"] = [f"{d.device_kind} x{len(devs)}" for d in devs][:1]
    FACTS["jax_sees_tpu"] = any(d.platform == "tpu" for d in devs)
    say(f"  ==> jax sees the TPU: {FACTS['jax_sees_tpu']}")
except Exception as e:
    FACTS["jax_sees_tpu"] = False
    say(f"  jax FAILED: {type(e).__name__}: {e}")

# ---- tensorflow is the other way in, and what the TPU plugin reports
try:
    import tensorflow as tf
    say(f"  {'tensorflow':<26} {tf.__version__}")
    tpus = tf.config.list_logical_devices("TPU")
    say(f"  {'tf logical TPUs':<26} {tpus}")
    res = tf.distribute.cluster_resolver.TPUClusterResolver()
    say(f"  {'tf cluster':<26} {res.cluster_spec().as_dict() if hasattr(res.cluster_spec(), 'as_dict') else res.cluster_spec()}")
    FACTS["tf_tpus"] = len(tpus)
except Exception as e:
    say(f"  tensorflow FAILED: {type(e).__name__}: {e}")

# ---- tinygrad has a TPU backend; the study cares whether it is reachable
try:
    from tinygrad import Device
    say(f"  {'tinygrad devices':<26} {[d for d in Device._devices if d]}")
    say(f"  {'tinygrad DEFAULT':<26} {Device.DEFAULT}")
    FACTS["tinygrad_default"] = Device.DEFAULT
except Exception as e:
    say(f"  tinygrad not importable ({type(e).__name__}: {e}) -- "
        f"pip install tinygrad to probe the TPU backend")

FACTS["versions"] = FACTS.get("versions", "")
lines = [
    "# TPU lane, Kaggle",
    "",
    f"- jax sees a TPU: **{FACTS.get('jax_sees_tpu')}**",
    f"- devices: {FACTS.get('jax_devices')}",
    f"- jax: {FACTS.get('versions')}",
    f"- tinygrad DEFAULT: {FACTS.get('tinygrad_default')}",
    "",
    "```",
    sh("ls -l /dev/accel* 2>&1"),
    "```",
]
open(os.path.join(OUT, "SUMMARY.md"), "w").write("\n".join(lines))
import json
json.dump(FACTS, open(os.path.join(OUT, "facts.json"), "w"), indent=2, default=str)
for ln in lines:
    say(ln)